# **EDA de Resistencias Antimicrobianas**

## 1. Introducción. Pregunta de la investigación

¿Qué factores clínicos, demográficos, hospitalarios y socioeconómicos están más asociados a la aparición de resistencia antimicrobiana (AMR) en los cultivos positivos? ¿Qué bacterias son las más resistentes? ¿ A qué antibióticos? Comparación con Europa, que tienen sistemas sanitarios diferentes.

## 2. Carga de datos 

Primero importamos las librerias que vamos a utilizar

In [2]:
import pandas as pd

Cargamos los dataset. Están en formato .csv separados por ",": Seleccionamos las columnas con las que trabajar y cambiamos los nombres de las columnas. Los datos son públicos, proporcionados por la Universidad de Stanford.

In [10]:

# Carga individual de cada archivo en una sola línea
import pandas as pd

cohort = pd.read_csv('../data/cohort.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded',
                                                             'order_proc_id_coded','ordering_mode', 
                                                             'culture_description','was_positive',
                                                               'organism', 'antibiotic', 'susceptibility'], 
                                                               na_values=['Null'])
demographics = pd.read_csv('../data/demographics.csv', sep=',',usecols=['anon_id', 'pat_enc_csn_id_coded', 
                                                                      'order_proc_id_coded', 'age','gender'],
                                                                      na_values=['Null'])
resistance = pd.read_csv('../data/microbial_resistance.csv', sep=',',  usecols=['anon_id', 'pat_enc_csn_id_coded', 
                                                                               'order_proc_id_coded',
                                                                               'organism', 'antibiotic',
                                                                               'resistant_time_to_culturetime'], na_values=['Null'])
ward = pd.read_csv('../data/ward_info.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded',
                                                            'hosp_ward_IP', 'hosp_ward_OP',
                                                             'hosp_ward_ER', 'hosp_ward_ICU'], na_values=['Null'])
adi = pd.read_csv('../data/adi_scores.csv', sep=',', usecols=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded',
                                                              'adi_score', 'adi_state_rank'], na_values=['Null'])


In [11]:
# Renombrar IDs
for df in [cohort, demographics, resistance, ward, adi]:
    df.rename(columns={
        'order_proc_id_coded': 'culture_id',
        'pat_enc_csn_id_coded': 'episode_id'
    }, inplace=True)

# Renombrar las variables de ward
ward.rename(columns={
    'hosp_ward_IP': 'IP',
    'hosp_ward_OP': 'OP',
    'hosp_ward_ER': 'ER',
    'hosp_ward_ICU': 'ICU'
}, inplace=True)

Decidimos que columnas nos interesan para nuestro EDA:

Todas estan unidas por un ``anon_id``, ``episode_id``y ``culture_id``. Que utilizaremos para unir los diferentes dataframes.

* **Cohort**: ordering_mode, culture_description, was_positive, organism, antibiotic, susceptibility
* **Demographics**: age, gender
* **Resistance**: organism, antibiotic, resistant_time_to_culturetime (days)
* **Ward**: hosp_ward_IP, hosp_ward_OP,hosp_ward_ER, hosp_ward_ICU
* **ADI**: adi_score, adi_state_rank

Definiciones relevantes: 

- **Unidad de análisis:** un cultivo (`culture_id`).
- **Población:** cultivos con `was_positive == True` y al menos un resultado de sensibilidad.
- **AMR:** el cultivo tiene al menos un antibiótico con `susceptibility == Resistant`. Los "Intermediate" se excluyen con la posibilidad de analizarse aparte.
- **`resistant_time_to_culturetime`:** días entre la confirmación de una resistencia previa y la orden del cultivo. Es un **predictor** (historial), nunca el desenlace.

## 3. Calidad de los datos

Vamos a revisar los datos que tenemos: nulos, ID_huerfanos, duplicados y que tipos de variables tenemos

In [12]:
#Tipo de variables y tamaño

tables = {
    "cohort": cohort,
    "demographics": demographics,
    "resistance": resistance,
    "ward": ward,
    "adi": adi,
}

for name, df in tables.items():
    print("Filas x columnas:", df.shape)
    print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
    print(df.dtypes.to_frame("dtype"))


Filas x columnas: (2241050, 9)
Memoria: 800.9 MB
                     dtype
anon_id                str
episode_id           int64
culture_id           int64
ordering_mode          str
culture_description    str
was_positive         int64
organism               str
antibiotic             str
susceptibility         str
Filas x columnas: (751075, 5)
Memoria: 106.3 MB
              dtype
anon_id         str
episode_id    int64
culture_id    int64
age             str
gender      float64
Filas x columnas: (2161648, 6)
Memoria: 458.9 MB
                               dtype
anon_id                          str
episode_id                     int64
culture_id                     int64
organism                         str
antibiotic                       str
resistant_time_to_culturetime  int64
Filas x columnas: (751075, 7)
Memoria: 79.4 MB
            dtype
anon_id       str
episode_id  int64
culture_id  int64
IP          int64
OP          int64
ER          int64
ICU         int64
Filas x column

In [ ]:
#NAs

for name, df in tables.items():
    null = pd.DataFrame({
        "n_null": df.isna().sum(),
        "%_null": (df.isna().mean() * 100).round(2),
    })
    print(f"\n===== Nulos: {name} =====")
    print(null)
 


===== Nulos: cohort =====
                     n_null  %_null
anon_id                   0    0.00
episode_id                0    0.00
culture_id                0    0.00
ordering_mode         73042    3.26
culture_description       0    0.00
was_positive              0    0.00
organism             632308   28.21
antibiotic           632308   28.21
susceptibility       634468   28.31

===== Nulos: demographics =====
            n_null  %_null
anon_id          0    0.00
episode_id       0    0.00
culture_id       0    0.00
age              0    0.00
gender         120    0.02

===== Nulos: resistance =====
                               n_null  %_null
anon_id                             0     0.0
episode_id                          0     0.0
culture_id                          0     0.0
organism                            0     0.0
antibiotic                          0     0.0
resistant_time_to_culturetime       0     0.0

===== Nulos: ward =====
            n_null  %_null
anon_id      

In [ ]:
#Duplicados

print("\n===== Duplicados: filas completamente idénticas =====")
for nombre, df in tables.items():
    print(f"{nombre}: {df.duplicated().sum():,} ({df.duplicated().mean() * 100:.2f}%)")
 
# Granularidad: filas por cultivo.
# Esperado: demographics, ward y adi -> 1 fila por cultivo.
#           cohort y resistance -> varias (organismo x antibiótico).
print("\n===== Filas por cultivo =====")
for nombre, df in tables.items():
    n = df.groupby("culture_id").size()
    print(f"\n{nombre}:")
    print(n.describe().round(2).to_frame("filas_por_cultivo").T)
    print("Cultivos con más de 1 fila:", (n > 1).sum())





===== Duplicados: filas completamente idénticas =====
cohort: 0 (0.00%)
demographics: 0 (0.00%)
resistance: 0 (0.00%)
ward: 0 (0.00%)
adi: 0 (0.00%)

===== Filas por cultivo =====

cohort:
                      count  mean   std  min  25%  50%  75%   max
filas_por_cultivo  751075.0  2.98  5.19  1.0  1.0  1.0  1.0  75.0
Cultivos con más de 1 fila: 118463

demographics:
                      count  mean  std  min  25%  50%  75%  max
filas_por_cultivo  751075.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
Cultivos con más de 1 fila: 0

resistance:
                      count   mean   std  min  25%  50%   75%    max
filas_por_cultivo  166537.0  12.98  29.1  1.0  2.0  4.0  11.0  483.0
Cultivos con más de 1 fila: 131521

ward:
                      count  mean  std  min  25%  50%  75%  max
filas_por_cultivo  751075.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
Cultivos con más de 1 fila: 0

adi:
                      count  mean  std  min  25%  50%  75%  max
filas_por_cultivo  751075.0   1.0  0.0  1.0  1.0 

131.521 de los 166.537 cultivos tienen varias filas (media 13, máximo 483). Esas filas no son duplicados, son resistencias previas distintas (organismo × antibiótico), y por eso las agregamos a una fila por cultivo.

In [25]:
# ---------------------------------------------------------------
# IDs únicos por tabla
# ---------------------------------------------------------------

IDS = ["anon_id", "culture_id", "episode_id"]
resumen_ids = pd.DataFrame(
    {nombre: {col: df[col].nunique() for col in IDS} for nombre, df in tables.items()}
).T
print("\n===== IDs únicos por tabla =====")
print(resumen_ids)
 
# ---------------------------------------------------------------
# 5. IDs huérfanos (id_culture) respecto a cohort
# ---------------------------------------------------------------
base = set(cohort["culture_id"].unique())
filas = []
for nombre, df in tables.items():
    if nombre == "cohort":
        continue
    ids = set(df["culture_id"].unique())
    filas.append({
        "tabla": nombre,
        "en_tabla_no_en_cohort": len(ids - base),
        "en_cohort_no_en_tabla": len(base - ids),
        "en_comun": len(ids & base),
    })
print("\n===== Huérfanos de id_culture frente a cohort =====")
print(pd.DataFrame(filas).set_index("tabla"))
 
# Lo mismo con anon_id
base_p = set(cohort["anon_id"].unique())
filas = []
for nombre, df in tables.items():
    if nombre == "cohort":
        continue
    ids = set(df["anon_id"].unique())
    filas.append({
        "tabla": nombre,
        "en_tabla_no_en_cohort": len(ids - base_p),
        "en_cohort_no_en_tabla": len(base_p - ids),
    })
print("\n===== Huérfanos de anon_id frente a cohort =====")
print(pd.DataFrame(filas).set_index("tabla"))
 
# ---------------------------------------------------------------
# 6. Consistencia de IDs: ¿cada id_culture pertenece a un único paciente
#    y a un único episodio, en todas las tablas?
# ---------------------------------------------------------------
print("\n===== Consistencia id_culture -> anon_id / id_enc =====")
for nombre, df in tables.items():
    n_pac = df.groupby("culture_id")["anon_id"].nunique()
    n_enc = df.groupby("culture_id")["episode_id"].nunique()
    print(f"{nombre}: cultivos con >1 paciente = {(n_pac > 1).sum()}, "
          f"cultivos con >1 episodio = {(n_enc > 1).sum()}")
 
# ---------------------------------------------------------------
# 7. Valores de las variables categóricas clave
# ---------------------------------------------------------------
print("\n===== Valores categóricos =====")
print(cohort["was_positive"].value_counts(dropna=False))
print(cohort["susceptibility"].value_counts(dropna=False))
print(cohort["ordering_mode"].value_counts(dropna=False))
print(cohort["culture_description"].value_counts(dropna=False).head(15))
print(demographics["age"].value_counts(dropna=False).sort_index())
print(demographics["gender"].value_counts(dropna=False))
print(adi[["adi_score", "adi_state_rank"]].describe())
 



===== IDs únicos por tabla =====
              anon_id  culture_id  episode_id
cohort         283715      751075      574584
demographics   283715      751075      574584
resistance      32780      166537      122726
ward           283715      751075      574584
adi            283715      751075      574584

===== Huérfanos de id_culture frente a cohort =====
              en_tabla_no_en_cohort  en_cohort_no_en_tabla  en_comun
tabla                                                               
demographics                      0                      0    751075
resistance                        0                 584538    166537
ward                              0                      0    751075
adi                               0                      0    751075

===== Huérfanos de anon_id frente a cohort =====
              en_tabla_no_en_cohort  en_cohort_no_en_tabla
tabla                                                     
demographics                      0                    

### **Resumen de los datos**
- 751.075 cultivos, 283.715 pacientes y 574.584 episodios.
+ Los dataframes de demographics, ward y adi tienen exactamente los mismos números de pacientes (anon_id) que cohort .
* Consistencia perfecta: cada cultivo pertenece a un único paciente y a un único episodio en todas las tablas.
- Edad sin nulos (nueve grupos)
+ Resistance: Subcojunto de 166.537 cultivos frente a 751.075 de cohort

Establecemos ``culture_id``como index. Es el identificador único y exclusivo de cada fila.

In [21]:
cohort.set_index('culture_id')
demographics.set_index('culture_id')
ward.set_index('culture_id')
adi.set_index('culture_id')
resistance.set_index('culture_id')

,anon_id,episode_id,organism,antibiotic,resistant_time_to_culturetime
culture_id,,,,,
701878721,JC1255618,131301644146,MYCOBACTERIUM ABSCESSUS,Minocycline,1266
737610687,JC533377,131311775703,KLEBSIELLA PNEUMONIAE,Ertapenem,845
517925931,JC669033,131207605914,BURKHOLDERIA CEPACIA,Minocycline,2231
570493265,JC1017583,131254647220,MYCOBACTERIUM IMMUNOGENUM,Linezolid,812
392129591,JC2465442,131016167664,PSEUDOMONAS AERUGINOSA,Colistin,210
...,...,...,...,...,...
496312476,JC1687848,131188621301,ENTEROCOCCUS FAECIUM,Vancomycin,129
636488681,JC1349716,131278316379,ENTEROCOCCUS SPECIES,Vancomycin,242
391557165,JC1374652,131016013539,ENTEROCOCCUS SPECIES,Vancomycin,63


En la cohorte 120000 cultivos fueron positivos para alguna bacteria. Es lo que vamos a utilizar para nuestro estudio.

In [23]:
print(cohort.groupby("culture_id")["was_positive"].max().value_counts())

was_positive
0    630798
1    120277
Name: count, dtype: int64


Antes de pasar a definir nuestra cohorte de estudio, vamos a proceder a modificar dos variables para que sean más claras de entender.

In [24]:

cohort['culture_result'] = cohort['was_positive'].map({0: 'Negative', 1: 'Positive'})
resistance.rename(columns={'resistant_time_to_culturetime': 'days_to_resistance_confirmation'}, inplace=True)


## 4. Creación de la cohorte de estudio

Vamos a incluir en nuestro estudio aquellos cultivos que presenten al menos una resistencia a un antibiótico. 

| Resultado Antibiograma | ¿AMR? | Interpretación EUCAST(2019)|
|---|---:|---|
| S | 0 | Susceptible |
| I | 0 | Susceptible con exposición aumentada |
| R | 1| Resistente |
| Synergism | excluir | No es categoría S/I/R |
| Inconclusive | excluir | Resultado no interpretable |
| Null | excluir | Sin resultado |